# PolERo LLM few-shot classifier (Ollama, localhost only)
Serves an open-weight model locally with Ollama on the 2 T4 GPUs and runs `python -m polero.llm`.
Few-shot examples come from fold 0 only, so outputs on folds 1-6 are valid OOF.
Run order: setup -> server -> (1) oof_sample to measure quality -> (2) test -> (3, optional) full oof for stacking.
Download `/kaggle/working/<exp>.zip` and unzip into `artifacts/` locally.

In [ ]:
# Settings: Kaggle -> Accelerator "GPU T4 x2", Internet ON.
import os, subprocess, sys
if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "-b", "member/giulia", "--depth", "1", "https://github.com/giulia223/Political-Evasion-Clarity-in-Romanian-Interviews.git", "repo"], check=True)
else:
    subprocess.run(["git", "-C", "repo", "pull"], check=True)
os.chdir("repo")
# Kaggle already has torch/transformers/sklearn/pandas; install only our package (no dependency changes).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
import torch; print(torch.__version__, torch.cuda.device_count(), "GPUs")

In [ ]:
# Install and start Ollama (localhost:11434). 2 GPUs are used automatically for big models.
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
env = dict(os.environ, OLLAMA_NUM_PARALLEL="8", OLLAMA_CONTEXT_LENGTH="16384", OLLAMA_KEEP_ALIVE="-1", OLLAMA_FLASH_ATTENTION="1")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
import time; time.sleep(8)
MODEL = "qwen3.6:35b-a3b"   # if this tag does not exist, try "qwen3:30b-a3b" (smaller/faster) or check ollama.com/library
r = subprocess.run(["ollama", "pull", MODEL]); print("pull rc", r.returncode)

In [ ]:
EXP = "llm_" + MODEL.replace(":", "_").replace(".", "")
base = ["--base-url", "http://localhost:11434/v1", "--model", MODEL, "--exp", EXP, "--workers", "8"]
# (1) quality check on a stratified sample (<=20 per class from folds 1-6), 1 sample each
subprocess.run([sys.executable, "-m", "polero.llm", *base, "--split", "oof_sample", "--n", "1"])

In [ ]:
# (2) test set, 3 samples each (self-consistency votes -> probabilities)
subprocess.run([sys.executable, "-m", "polero.llm", *base, "--split", "test", "--n", "3"])

In [ ]:
# (3) optional, long: all of folds 1-6 (~2800 rows) for stacking with the encoder
# subprocess.run([sys.executable, "-m", "polero.llm", *base, "--split", "oof", "--n", "1"])

In [ ]:
import shutil; shutil.make_archive(f"/kaggle/working/{EXP}", "zip", f"artifacts/{EXP}")
print("download /kaggle/working/" + EXP + ".zip")